<a href="https://colab.research.google.com/github/Vang004/Big_Data/blob/main/Fix_28042026_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import math
import time
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv          # pip install python-dotenv

# ── Load biến môi trường từ .env (KHÔNG hardcode key vào đây!) ──
load_dotenv()

# Các API Key lấy từ .env – nếu dùng API miễn phí thì để trống
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY", "")  # tuỳ chọn
OPENAQ_API_KEY      = os.getenv("OPENAQ_API_KEY", "")       # tuỳ chọn (v3)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType, DoubleType, TimestampType
)
from pyspark.sql.functions import (
    col, when, udf, to_timestamp,
    avg, max as F_max, min as F_min, stddev, count,
    hour, dayofmonth, month, dayofweek, greatest, lit,
    round as F_round, corr, percentile_approx, date_format
)
print("Thành công\n")

Thành công



In [2]:
# ============================================================
#  PHẦN 0 – KHỞI TẠO MÔI TRƯỜNG
# ============================================================

print("=" * 60)
print("  BIG DATA PIPELINE – CHẤT LƯỢNG KHÔNG KHÍ VIỆT NAM 2024")
print("=" * 60)

# Spark Session
print("\n⏳ Khởi tạo SparkSession...")
spark = (
    SparkSession.builder
    .appName("VN_AirQuality_BigData_2024")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} sẵn sàng")

# Thư mục lưu trữ
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/BigData_AirQuality_VN"
    print("✅ Google Drive đã kết nối!")
except ImportError:
    BASE_DIR = "/tmp/BigData_AirQuality_VN"
    print("⚠️  Chạy trên môi trường local / GitHub Actions.")

for sub in ["raw", "clean", "enriched", "analytics", "reports"]:
    os.makedirs(f"{BASE_DIR}/{sub}", exist_ok=True)
print(f"📂 BASE_DIR = {BASE_DIR}")

assert spark is not None,         "❌ SparkSession chưa khởi tạo!"
assert os.path.isdir(BASE_DIR),   "❌ Thư mục BASE_DIR không tồn tại!"
print("🎉 PASSED PHẦN 0\n")

  BIG DATA PIPELINE – CHẤT LƯỢNG KHÔNG KHÍ VIỆT NAM 2024

⏳ Khởi tạo SparkSession...
✅ Spark 4.0.2 sẵn sàng
Mounted at /content/drive
✅ Google Drive đã kết nối!
📂 BASE_DIR = /content/drive/MyDrive/BigData_AirQuality_VN
🎉 PASSED PHẦN 0



In [3]:
# ============================================================
#  PHẦN 1 – INGESTION: THU THẬP DỮ LIỆU QUA API
# ============================================================
#
#  Nguồn dữ liệu (100 % miễn phí, không cần key):
#  • Open-Meteo Archive API  – thời tiết lịch sử theo giờ
#  • OpenAQ API v2           – đo thực tế PM2.5/PM10/CO/NO2 tại VN
#
#  Dữ liệu bổ sung mô phỏng Big Data:
#  • 50 trạm × 365 ngày × 24h ≈ 438,000 bản ghi
# ============================================================

# ── 1.1 Danh sách 50 trạm quan trắc ──────────────────────────
STATIONS = [
    # Hà Nội – 10 trạm
    {"id":"HN01",  "name":"HN – Hoàn Kiếm",      "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0285, "lon":105.8542, "type":"urban"},
    {"id":"HN02",  "name":"HN – Cầu Giấy",        "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0333, "lon":105.7833, "type":"suburban"},
    {"id":"HN03",  "name":"HN – Long Biên",        "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0500, "lon":105.9000, "type":"industrial"},
    {"id":"HN04",  "name":"HN – Đống Đa",          "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0245, "lon":105.8412, "type":"urban"},
    {"id":"HN05",  "name":"HN – Hoàng Mai",        "city":"Hà Nội",       "province":"Hà Nội",       "lat":20.9833, "lon":105.8500, "type":"suburban"},
    {"id":"HN06",  "name":"HN – Bắc Từ Liêm",     "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0833, "lon":105.7833, "type":"suburban"},
    {"id":"HN07",  "name":"HN – Thanh Xuân",       "city":"Hà Nội",       "province":"Hà Nội",       "lat":20.9944, "lon":105.8095, "type":"urban"},
    {"id":"HN08",  "name":"HN – Mê Linh",          "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.2000, "lon":105.7500, "type":"rural"},
    {"id":"HN09",  "name":"HN – Đông Anh (KCN)",   "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.1500, "lon":105.8500, "type":"industrial"},
    {"id":"HN10",  "name":"HN – Gia Lâm",          "city":"Hà Nội",       "province":"Hà Nội",       "lat":21.0167, "lon":105.9333, "type":"suburban"},
    # Hồ Chí Minh – 8 trạm
    {"id":"HCM01", "name":"HCM – Quận 1",          "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.7769, "lon":106.7009, "type":"urban"},
    {"id":"HCM02", "name":"HCM – Bình Thạnh",      "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.8000, "lon":106.7167, "type":"urban"},
    {"id":"HCM03", "name":"HCM – Gò Vấp",          "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.8333, "lon":106.6833, "type":"suburban"},
    {"id":"HCM04", "name":"HCM – Thủ Đức (KCN)",   "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.8667, "lon":106.7500, "type":"industrial"},
    {"id":"HCM05", "name":"HCM – Nhà Bè",          "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.6833, "lon":106.7500, "type":"suburban"},
    {"id":"HCM06", "name":"HCM – Cần Giờ (rừng)",  "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.4000, "lon":106.9500, "type":"rural"},
    {"id":"HCM07", "name":"HCM – Quận 9",          "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.8500, "lon":106.7833, "type":"suburban"},
    {"id":"HCM08", "name":"HCM – Bình Chánh",      "city":"Hồ Chí Minh",  "province":"Hồ Chí Minh",  "lat":10.6833, "lon":106.6000, "type":"suburban"},
    # Đà Nẵng – 4 trạm
    {"id":"DN01",  "name":"ĐN – Hải Châu",         "city":"Đà Nẵng",      "province":"Đà Nẵng",      "lat":16.0544, "lon":108.2022, "type":"urban"},
    {"id":"DN02",  "name":"ĐN – Liên Chiểu (KCN)", "city":"Đà Nẵng",      "province":"Đà Nẵng",      "lat":16.0833, "lon":108.1333, "type":"industrial"},
    {"id":"DN03",  "name":"ĐN – Ngũ Hành Sơn",     "city":"Đà Nẵng",      "province":"Đà Nẵng",      "lat":16.0000, "lon":108.2500, "type":"suburban"},
    {"id":"DN04",  "name":"ĐN – Sơn Trà",          "city":"Đà Nẵng",      "province":"Đà Nẵng",      "lat":16.1000, "lon":108.2333, "type":"urban"},
    # Hải Phòng – 2 trạm
    {"id":"HP01",  "name":"HP – Hồng Bàng",        "city":"Hải Phòng",    "province":"Hải Phòng",    "lat":20.8449, "lon":106.6881, "type":"urban"},
    {"id":"HP02",  "name":"HP – KCN Đình Vũ",      "city":"Hải Phòng",    "province":"Hải Phòng",    "lat":20.8000, "lon":106.8000, "type":"industrial"},
    # Quảng Ninh – 2 trạm
    {"id":"QN01",  "name":"QN – Hạ Long",          "city":"Hạ Long",      "province":"Quảng Ninh",   "lat":20.9517, "lon":107.0800, "type":"industrial"},
    {"id":"QN02",  "name":"QN – Cẩm Phả",          "city":"Cẩm Phả",     "province":"Quảng Ninh",   "lat":21.0200, "lon":107.3200, "type":"industrial"},
    # Miền Trung
    {"id":"HUE01", "name":"Huế – TP Huế",          "city":"Huế",          "province":"TT-Huế",       "lat":16.4637, "lon":107.5909, "type":"urban"},
    {"id":"QNG01", "name":"Quảng Ngãi",             "city":"Quảng Ngãi",   "province":"Quảng Ngãi",   "lat":15.1200, "lon":108.8000, "type":"urban"},
    {"id":"NT01",  "name":"Nha Trang",              "city":"Nha Trang",    "province":"Khánh Hòa",    "lat":12.2388, "lon":109.1967, "type":"urban"},
    {"id":"PY01",  "name":"Phú Yên – Tuy Hòa",     "city":"Tuy Hòa",     "province":"Phú Yên",      "lat":13.0967, "lon":109.3228, "type":"urban"},
    # Miền Nam – công nghiệp
    {"id":"BD01",  "name":"Bình Dương – VSIP",      "city":"Thủ Dầu Một", "province":"Bình Dương",   "lat":10.9804, "lon":106.6519, "type":"industrial"},
    {"id":"BD02",  "name":"Bình Dương – Mỹ Phước",  "city":"Bến Cát",     "province":"Bình Dương",   "lat":11.0667, "lon":106.6667, "type":"industrial"},
    {"id":"DNA01", "name":"Đồng Nai – Biên Hòa",    "city":"Biên Hòa",    "province":"Đồng Nai",     "lat":10.9574, "lon":106.8426, "type":"industrial"},
    {"id":"VT01",  "name":"Vũng Tàu",               "city":"Vũng Tàu",    "province":"BR-VT",        "lat":10.3460, "lon":107.0843, "type":"urban"},
    # Đồng bằng sông Cửu Long
    {"id":"CT01",  "name":"Cần Thơ – Ninh Kiều",    "city":"Cần Thơ",     "province":"Cần Thơ",      "lat":10.0452, "lon":105.7469, "type":"urban"},
    {"id":"CT02",  "name":"Cần Thơ – Ô Môn (CN)",   "city":"Cần Thơ",     "province":"Cần Thơ",      "lat":10.0833, "lon":105.6500, "type":"industrial"},
    {"id":"AG01",  "name":"An Giang – Long Xuyên",   "city":"Long Xuyên",  "province":"An Giang",     "lat":10.3833, "lon":105.4333, "type":"urban"},
    {"id":"KG01",  "name":"Kiên Giang – Rạch Giá",  "city":"Rạch Giá",   "province":"Kiên Giang",   "lat":10.0133, "lon":105.0900, "type":"urban"},
    {"id":"TG01",  "name":"Tiền Giang – Mỹ Tho",    "city":"Mỹ Tho",     "province":"Tiền Giang",   "lat":10.3600, "lon":106.3600, "type":"urban"},
    {"id":"VL01",  "name":"Vĩnh Long",               "city":"Vĩnh Long",   "province":"Vĩnh Long",    "lat":10.2500, "lon":105.9667, "type":"urban"},
    {"id":"CM01",  "name":"Cà Mau – Đất Mũi",        "city":"Cà Mau",     "province":"Cà Mau",       "lat": 9.1833, "lon":105.1500, "type":"rural"},
    # Tây Nguyên
    {"id":"DK01",  "name":"Đắk Lắk – BMT",          "city":"Buôn Ma Thuột","province":"Đắk Lắk",    "lat":12.6667, "lon":108.0500, "type":"urban"},
    {"id":"GL01",  "name":"Gia Lai – Pleiku",        "city":"Pleiku",      "province":"Gia Lai",      "lat":13.9833, "lon":108.0000, "type":"urban"},
    {"id":"LD01",  "name":"Lâm Đồng – Đà Lạt",      "city":"Đà Lạt",     "province":"Lâm Đồng",     "lat":11.9400, "lon":108.4500, "type":"urban"},
    # Miền Bắc khác
    {"id":"TH01",  "name":"Thanh Hóa",               "city":"Thanh Hóa",   "province":"Thanh Hóa",    "lat":19.8000, "lon":105.7833, "type":"urban"},
    {"id":"NA01",  "name":"Nghệ An – Vinh",           "city":"Vinh",        "province":"Nghệ An",      "lat":18.6795, "lon":105.6882, "type":"urban"},
    {"id":"SL01",  "name":"Sơn La",                   "city":"Sơn La",      "province":"Sơn La",       "lat":21.3250, "lon":103.9167, "type":"urban"},
    {"id":"LC01",  "name":"Lào Cai",                  "city":"Lào Cai",    "province":"Lào Cai",      "lat":22.4833, "lon":103.9667, "type":"urban"},
    {"id":"BH01",  "name":"Phan Thiết",               "city":"Phan Thiết",  "province":"Bình Thuận",   "lat":10.9333, "lon":108.1000, "type":"urban"},
]
print(f"📍 Đã nạp {len(STATIONS)} trạm quan trắc trên toàn quốc")


📍 Đã nạp 49 trạm quan trắc trên toàn quốc


In [4]:
# ── 1.2 Hàm gọi Open-Meteo Archive API (miễn phí, không cần key) ──
def fetch_openmeteo_year(lat: float, lon: float,
                          start: str = "2024-01-01",
                          end:   str = "2024-12-31") -> dict | None:
    """
    Gọi Open-Meteo Archive API để lấy dữ liệu thời tiết theo giờ.
    MIỄN PHÍ – không cần API key.
    Endpoint: https://archive-api.open-meteo.com/v1/archive
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":    lat,
        "longitude":   lon,
        "start_date":  start,
        "end_date":    end,
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "precipitation",
            "surface_pressure",
            "shortwave_radiation",
        ]),
        "timezone":        "Asia/Bangkok",
        "wind_speed_unit": "ms",
    }
    try:
        r = requests.get(url, params=params, timeout=45)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"   ⚠️  Open-Meteo lỗi ({lat},{lon}): {e}")
        return None
    print("PASS THÀNH CÔNG")

In [5]:
# ── 1.3 Hàm gọi OpenAQ API v2 (miễn phí, rate-limit 10 req/s) ───
import requests

# Dán mã bạn vừa lấy được vào đây
MY_API_KEY = "250f3bd85497444efaa20bab4e3af9e41d60965b"
def fetch_openaq_realdata(country: str = "VN",
                           parameter: str = "pm25",
                           limit: int = 3000) -> list[dict]:
    """
    Gọi OpenAQ v2 lấy các phép đo thực tế tại Việt Nam.
    MIỄN PHÍ – không cần key với v2 (tốc độ giới hạn 10 req/s).
    Nếu muốn dùng v3 (nhanh hơn), lấy key tại: https://openaq.org
    và thêm OPENAQ_API_KEY vào file .env
    """
    url = "https://api.openaq.org/v3/measurements"
    params = {
        "country":   country,
        "parameter": parameter,
        "limit":     limit,
        "sort":      "desc",
        "order_by":  "datetime",
    }
    headers = {"Accept": "application/json"}
    # Nếu có API key v3, dùng endpoint v3 + header X-API-Key
    if OPENAQ_API_KEY:
        headers["X-API-Key"] = OPENAQ_API_KEY
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        r.raise_for_status()
        results = r.json().get("results", [])
        print(f"   ✅ OpenAQ [{parameter}]: {len(results)} bản ghi")
        return results
    except Exception as e:
        print(f"   ⚠️  OpenAQ lỗi [{parameter}]: {e}")
        return []
print("🎉 PASSED THÀNH CÔNG\n")

🎉 PASSED THÀNH CÔNG



In [6]:
# ── 1.4 Thu thập dữ liệu thời tiết thực từ Open-Meteo ──────────
print("\n📡 Bước 1: Lấy dữ liệu thời tiết 1 năm từ Open-Meteo (5 thành phố)…")
REPRESENTATIVE = [
    {"id":"HN01",  "lat":21.0285, "lon":105.8542},  # Hà Nội
    {"id":"HCM01", "lat":10.7769, "lon":106.7009},  # HCM
    {"id":"DN01",  "lat":16.0544, "lon":108.2022},  # Đà Nẵng
    {"id":"CT01",  "lat":10.0452, "lon":105.7469},  # Cần Thơ
    {"id":"HP01",  "lat":20.8449, "lon":106.6881},  # Hải Phòng
]

raw_weather = {}
for sta in REPRESENTATIVE:
    print(f"   Đang tải: {sta['id']}…", end=" ", flush=True)
    data = fetch_openmeteo_year(sta["lat"], sta["lon"])
    if data and "hourly" in data:
        raw_weather[sta["id"]] = data["hourly"]
        n = len(data["hourly"]["time"])
        print(f"✅  {n:,} điểm dữ liệu")
    else:
        print("⚠️  Không lấy được")
    time.sleep(0.5)   # Tránh spam API

# Lưu raw weather JSON
weather_path = f"{BASE_DIR}/raw/openmeteo_weather.json"
with open(weather_path, "w", encoding="utf-8") as f:
    json.dump(raw_weather, f)
print(f"   💾 Đã lưu: {weather_path}")
print("🎉 PASSED THÀNH CÔNG\n")


📡 Bước 1: Lấy dữ liệu thời tiết 1 năm từ Open-Meteo (5 thành phố)…
   Đang tải: HN01… ✅  8,784 điểm dữ liệu
   Đang tải: HCM01… ✅  8,784 điểm dữ liệu
   Đang tải: DN01… ✅  8,784 điểm dữ liệu
   Đang tải: CT01… ✅  8,784 điểm dữ liệu
   Đang tải: HP01… ✅  8,784 điểm dữ liệu
   💾 Đã lưu: /content/drive/MyDrive/BigData_AirQuality_VN/raw/openmeteo_weather.json
🎉 PASSED THÀNH CÔNG



In [7]:
# --- THÔNG TIN CẤU HÌNH ---
API_KEY = "83f9965f14e9aecfccd757c9550feb38"

# ── 1.5 Thu thập dữ liệu chất lượng không khí từ OpenWeather ───
print("\n📡 Bước 2: Lấy dữ liệu chất lượng không khí thực từ OpenWeather...")

air_pollution_records = []

# Sử dụng lại danh sách REPRESENTATIVE từ Bước 1.4 để đồng bộ tọa độ
for sta in REPRESENTATIVE:
    sid = sta["id"]
    lat = sta["lat"]
    lon = sta["lon"]

    print(f"   Đang lấy dữ liệu ô nhiễm: {sid}...", end=" ", flush=True)

    # API URL của OpenWeather Air Pollution
    url = f"http://api.openweathermap.org/data/2.5/air_pollution?lat={lat}&lon={lon}&appid={API_KEY}"

    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()

            # OpenWeather trả về một list các bản ghi (thường là 1 cho hiện tại)
            for item in data.get("list", []):
                components = item.get("components", {})

                # Cấu trúc hóa dữ liệu để lưu vào Data Lake
                air_pollution_records.append({
                    "station_id": sid,
                    "aqi": item.get("main", {}).get("aqi"), # Chỉ số chất lượng không khí (1-5)
                    "pm2_5": components.get("pm2_5"),
                    "pm10": components.get("pm10"),
                    "co": components.get("co"),
                    "no2": components.get("no2"),
                    "so2": components.get("so2"),
                    "o3": components.get("o3"),
                    "timestamp": item.get("dt"), # Unix timestamp
                    "lat": lat,
                    "lon": lon,
                    "unit": "ug/m3"
                })
            print(f"✅")
        else:
            print(f"⚠️ Lỗi {response.status_code}")

    except Exception as e:
        print(f"❌ Lỗi kết nối: {str(e)}")

    time.sleep(0.5) # Tránh spam API (Rate-limit)

# Đường dẫn lưu file (Đồng bộ với BASE_DIR của bạn)
openaq_path = f"{BASE_DIR}/raw/openaq_realdata.json"

with open(openaq_path, "w", encoding="utf-8") as f:
    json.dump(air_pollution_records, f, ensure_ascii=False, indent=4)

print(f"\n   💾 Đã lưu {len(air_pollution_records):,} bản ghi Air Pollution → {openaq_path}")
print("🎉 PASSED THÀNH CÔNG\n")


📡 Bước 2: Lấy dữ liệu chất lượng không khí thực từ OpenWeather...
   Đang lấy dữ liệu ô nhiễm: HN01... ✅
   Đang lấy dữ liệu ô nhiễm: HCM01... ✅
   Đang lấy dữ liệu ô nhiễm: DN01... ✅
   Đang lấy dữ liệu ô nhiễm: CT01... ✅
   Đang lấy dữ liệu ô nhiễm: HP01... ✅

   💾 Đã lưu 5 bản ghi Air Pollution → /content/drive/MyDrive/BigData_AirQuality_VN/raw/openaq_realdata.json
🎉 PASSED THÀNH CÔNG



In [8]:
# ── 1.6 Sinh dữ liệu quy mô lớn (BIG DATA ~438,000 bản ghi) ────
def build_bigdata_dataset(stations: list[dict],
                           weather_ref: dict,
                           start: str = "2024-01-01",
                           end:   str = "2024-12-31") -> list[dict]:
    """
    Tạo dataset quy mô lớn: 50 trạm × 365 ngày × 24 giờ ≈ 438,000 bản ghi.

    Mô hình thống kê:
      • Giá trị nền theo loại trạm (urban / industrial / suburban / rural)
      • Hệ số rush-hour (7–9h, 17–19h: +45%)
      • Hệ số mùa khô / mưa (+30% mùa khô)
      • Chu kỳ ozone: đỉnh vào 11–14h (bức xạ mặt trời)
      • Log-normal noise mô phỏng biến động thực tế
      • Hiệu chỉnh nhiệt độ / độ ẩm từ dữ liệu Open-Meteo thực
    """
    # Nồng độ nền theo loại trạm (µg/m³) – Tham chiếu WHO + VN thực tế
    BASE = {
        #            PM2.5  PM10   CO    NO2   SO2    O3
        "urban":    (38,    60,    1.4,  55,   22,   0.045),
        "industrial":(70,  115,    2.8,  90,   60,   0.030),
        "suburban": (25,    42,    0.9,  35,   14,   0.055),
        "rural":    (12,    22,    0.4,  18,    6,   0.065),
    }

    # Climate profile từ Open-Meteo
    hn_temp, hcm_temp = {}, {}
    if "HN01" in weather_ref:
        df_ref = pd.DataFrame(weather_ref["HN01"])
        df_ref["month"] = pd.to_datetime(df_ref["time"]).dt.month
        hn_temp = df_ref.groupby("month")["temperature_2m"].mean().to_dict()
    if "HCM01" in weather_ref:
        df_ref = pd.DataFrame(weather_ref["HCM01"])
        df_ref["month"] = pd.to_datetime(df_ref["time"]).dt.month
        hcm_temp = df_ref.groupby("month")["temperature_2m"].mean().to_dict()

    rng = np.random.default_rng(42)
    records = []
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt   = datetime.strptime(end,   "%Y-%m-%d")

    for station in stations:
        stype = station.get("type", "urban")
        pm25_b, pm10_b, co_b, no2_b, so2_b, o3_b = BASE[stype]
        is_north = station["lat"] >= 16
        temp_ref = hn_temp if is_north else hcm_temp

        for day_offset in range((end_dt - start_dt).days + 1):
            day = start_dt + timedelta(days=day_offset)
            m   = day.month
            season_f = (1.30 if m in [11,12,1,2,3]  else 0.82) if is_north \
                  else (1.20 if m in [12,1,2,3,4]   else 0.88)
            t_ref = temp_ref.get(m, 20 + 8 * math.sin(2*math.pi*(m-3)/12))

            for h in range(24):
                ts = day.replace(hour=h)
                is_rush   = h in [7,8,9,17,18,19]
                is_night  = h in [1,2,3,4]
                is_midday = h in [11,12,13,14]

                traffic_f = 1.45 if is_rush else (0.65 if is_night else 1.0)
                ozone_f   = 1.60 if is_midday else 0.70
                hour_f    = traffic_f * season_f

                def noisy(base, std=0.22):
                    return round(float(max(0.0,
                        base * hour_f * rng.lognormal(0, std))), 2)

                records.append({
                    "station_id":   station["id"],
                    "station_name": station["name"],
                    "city":         station["city"],
                    "province":     station["province"],
                    "station_type": station["type"],
                    "lat":          station["lat"],
                    "lon":          station["lon"],
                    "timestamp":    ts.strftime("%Y-%m-%d %H:%M:%S"),
                    "pm25":  noisy(pm25_b),
                    "pm10":  noisy(pm10_b),
                    "co":    round(float(max(0.0, co_b * hour_f * rng.lognormal(0, 0.18))), 3),
                    "no2":   noisy(no2_b),
                    "so2":   noisy(so2_b),
                    "o3":    round(float(max(0.0, o3_b * ozone_f * season_f * rng.lognormal(0, 0.20))), 4),
                    "temperature": round(t_ref + rng.uniform(-4,4) + 4*math.sin(2*math.pi*(h-6)/24), 1),
                    "humidity":    round(float(max(10, min(100, 70+15*math.sin(2*math.pi*(m-7)/12)+rng.uniform(-12,12)))), 1),
                    "wind_speed":  round(float(max(0, rng.exponential(2.3))), 1),
                    "pressure":    round(1013.0 + rng.uniform(-6, 6), 1),
                    "radiation":   round(float(max(0, 800*math.sin(math.pi*(h-6)/12)+rng.uniform(-50,50))) if 6<=h<=18 else 0.0, 1),
                })
    return records


print(f"\n🏭 Sinh dataset: {len(STATIONS)} trạm × 365 ngày × 24h ≈ 438,000 bản ghi…")
t0 = time.time()
bigdata = build_bigdata_dataset(STATIONS, raw_weather)
print(f"   ✅ {len(bigdata):,} bản ghi ({time.time()-t0:.1f}s)")

raw_json_path = f"{BASE_DIR}/raw/air_quality_raw.json"
with open(raw_json_path, "w", encoding="utf-8") as f:
    json.dump(bigdata, f)
print(f"   💾 Raw JSON → {raw_json_path}")

# ── Auto-test 1 ───────────────────────────────────────────────
assert os.path.exists(raw_json_path), "❌ File raw JSON chưa được tạo!"
assert len(bigdata) > 400_000,        "❌ Dataset chưa đủ quy mô Big Data (>400,000)!"
assert "pm25"      in bigdata[0],     "❌ Thiếu trường pm25!"
assert "timestamp" in bigdata[0],     "❌ Thiếu trường timestamp!"
print(f"🎉 PASSED PHẦN 1 – {len(bigdata):,} bản ghi sẵn sàng\n")


🏭 Sinh dataset: 49 trạm × 365 ngày × 24h ≈ 438,000 bản ghi…
   ✅ 430,416 bản ghi (19.9s)
   💾 Raw JSON → /content/drive/MyDrive/BigData_AirQuality_VN/raw/air_quality_raw.json
🎉 PASSED PHẦN 1 – 430,416 bản ghi sẵn sàng



In [9]:
# ============================================================
#  PHẦN 2 – STORAGE & PROCESSING: LÀM SẠCH DỮ LIỆU PYSPARK
#  → Lưu Parquet (định dạng chuẩn Big Data)
# ============================================================

SCHEMA = StructType([
    StructField("station_id",   StringType(), True),
    StructField("station_name", StringType(), True),
    StructField("city",         StringType(), True),
    StructField("province",     StringType(), True),
    StructField("station_type", StringType(), True),
    StructField("lat",          DoubleType(), True),
    StructField("lon",          DoubleType(), True),
    StructField("timestamp",    StringType(), True),
    StructField("pm25",         DoubleType(), True),
    StructField("pm10",         DoubleType(), True),
    StructField("co",           DoubleType(), True),
    StructField("no2",          DoubleType(), True),
    StructField("so2",          DoubleType(), True),
    StructField("o3",           DoubleType(), True),
    StructField("temperature",  DoubleType(), True),
    StructField("humidity",     DoubleType(), True),
    StructField("wind_speed",   DoubleType(), True),
    StructField("pressure",     DoubleType(), True),
    StructField("radiation",    DoubleType(), True),
])

print("📥 Đọc dữ liệu vào Spark…")
df_raw = spark.read.schema(SCHEMA).json(raw_json_path)
print(f"   Tổng bản ghi thô: {df_raw.count():,}")

# Parse timestamp
df = df_raw.withColumn("timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

# Điền NULL bằng trung vị theo từng trạm
print("   🔧 Xử lý giá trị NULL (median imputation)…")
pollutants = ["pm25","pm10","co","no2","so2","o3","temperature","humidity","wind_speed","pressure"]
stats = df.groupBy("station_id").agg(
    *[F.percentile_approx(p, 0.5).alias(f"med_{p}") for p in pollutants]
)
df = df.join(stats, on="station_id", how="left")
for p in pollutants:
    df = df.withColumn(p, when(col(p).isNull(), col(f"med_{p}")).otherwise(col(p)))
df = df.drop(*[f"med_{p}" for p in pollutants])

# Lọc giá trị phi lý (theo chuẩn WHO)
print("   🔧 Lọc giá trị phi lý…")
df_clean = df.filter(
    (col("pm25")        >= 0)   & (col("pm25")        <= 1000) &
    (col("pm10")        >= 0)   & (col("pm10")        <= 2000) &
    (col("co")          >= 0)   & (col("co")          <=   50) &
    (col("no2")         >= 0)   & (col("no2")         <= 2050) &
    (col("so2")         >= 0)   & (col("so2")         <= 1005) &
    (col("o3")          >= 0)   & (col("o3")          <=  0.61) &
    (col("temperature") >= -10) & (col("temperature") <=   50) &
    (col("humidity")    >=  0)  & (col("humidity")    <=  100) &
    (col("pressure")    >= 900) & (col("pressure")    <= 1100) &
    col("timestamp").isNotNull()
)

before = df_clean.count()
df_clean = df_clean.dropDuplicates(["station_id","timestamp"])
after   = df_clean.count()
print(f"   ✅ Sau làm sạch: {after:,} bản ghi (loại {before-after:,} trùng/lỗi)")

# ▶ LƯU PARQUET – định dạng chuẩn Big Data (có thể đẩy lên HDFS/MinIO)
clean_path = f"{BASE_DIR}/clean/air_quality_clean.parquet"
df_clean.write.mode("overwrite").parquet(clean_path)
print(f"   💾 Clean data (Parquet) → {clean_path}")

# ── Auto-test 2 ───────────────────────────────────────────────
assert df_clean is not None,           "❌ df_clean chưa được tạo!"
assert df_clean.count() > 0,           "❌ DataFrame rỗng!"
assert "timestamp" in df_clean.columns,"❌ Thiếu cột timestamp!"
null_pm25 = df_clean.filter(col("pm25").isNull()).count()
assert null_pm25 == 0,                 "❌ Vẫn còn NULL trong pm25!"
print("🎉 PASSED PHẦN 2\n")

📥 Đọc dữ liệu vào Spark…
   Tổng bản ghi thô: 430,416
   🔧 Xử lý giá trị NULL (median imputation)…
   🔧 Lọc giá trị phi lý…
   ✅ Sau làm sạch: 430,416 bản ghi (loại 0 trùng/lỗi)
   💾 Clean data (Parquet) → /content/drive/MyDrive/BigData_AirQuality_VN/clean/air_quality_clean.parquet
🎉 PASSED PHẦN 2



In [10]:
# ============================================================
#  PHẦN 3 – ENRICHMENT: TÍNH AQI CHUẨN US EPA
#
#  Công thức Piecewise Linear Interpolation:
#
#         (I_Hi – I_Lo)
#  AQI = ────────────── × (C_p – BP_Lo) + I_Lo
#         (BP_Hi – BP_Lo)
#
#  Ref: https://document.airnow.gov/technical-assistance-document-...
# ============================================================

def _interp(c, bps):
    """Nội suy tuyến tính theo bảng breakpoints EPA."""
    for (bp_lo, bp_hi, i_lo, i_hi) in bps:
        if bp_lo <= c <= bp_hi:
            return int(round((i_hi-i_lo)/(bp_hi-bp_lo)*(c-bp_lo)+i_lo))
    return 500 if c > bps[-1][1] else 0

@udf(IntegerType())
def aqi_pm25(c):
    """AQI PM2.5 (µg/m³, 24-hr avg) – làm tròn 1 chữ số thập phân."""
    if c is None or c < 0: return None
    c = round(c, 1)
    return _interp(c, [
        (0.0,   12.0,   0,  50), (12.1,  35.4,  51, 100),
        (35.5,  55.4, 101, 150), (55.5, 150.4, 151, 200),
        (150.5,250.4, 201, 300), (250.5,350.4, 301, 400),
        (350.5,500.4, 401, 500),
    ])

@udf(IntegerType())
def aqi_pm10(c):
    """AQI PM10 (µg/m³, 24-hr avg) – truncate trước khi tra."""
    if c is None or c < 0: return None
    c = int(c)
    return _interp(c, [
        (0,   54,   0,  50), (55,  154,  51, 100),
        (155, 254, 101, 150), (255, 354, 151, 200),
        (355, 424, 201, 300), (425, 504, 301, 400),
        (505, 604, 401, 500),
    ])

@udf(IntegerType())
def aqi_co(c):
    """AQI CO (ppm, 8-hr avg) – làm tròn 1 chữ số."""
    if c is None or c < 0: return None
    c = round(c, 1)
    return _interp(c, [
        (0.0,  4.4,   0,  50), (4.5,   9.4,  51, 100),
        (9.5,  12.4, 101, 150), (12.5,  15.4, 151, 200),
        (15.5, 30.4, 201, 300), (30.5,  40.4, 301, 400),
        (40.5, 50.4, 401, 500),
    ])

@udf(IntegerType())
def aqi_no2(c):
    """AQI NO₂ (ppb, 1-hr avg) – truncate."""
    if c is None or c < 0: return None
    c = int(c)
    return _interp(c, [
        (0,    53,   0,  50), (54,   100,  51, 100),
        (101,  360, 101, 150), (361,  649, 151, 200),
        (650, 1249, 201, 300), (1250,1649, 301, 400),
        (1650,2049, 401, 500),
    ])

@udf(IntegerType())
def aqi_so2(c):
    """AQI SO₂ (ppb, 1-hr avg) – truncate."""
    if c is None or c < 0: return None
    c = int(c)
    return _interp(c, [
        (0,   35,   0,  50), (36,   75,  51, 100),
        (76,  185, 101, 150), (186,  304, 151, 200),
        (305, 604, 201, 300), (605,  804, 301, 400),
        (805,1004, 401, 500),
    ])

@udf(IntegerType())
def aqi_o3(c):
    """AQI O₃ (ppm, 8-hr avg) – làm tròn 3 chữ số."""
    if c is None or c < 0: return None
    c = round(c, 3)
    return _interp(c, [
        (0.000,0.054,   0,  50), (0.055,0.070,  51, 100),
        (0.071,0.085, 101, 150), (0.086,0.105, 151, 200),
        (0.106,0.200, 201, 300), (0.405,0.504, 301, 400),
        (0.505,0.604, 401, 500),
    ])

@udf(StringType())
def aqi_level(aqi):
    """Phân loại chất lượng không khí (US EPA + TCVN)."""
    if aqi is None: return "Không xác định"
    if aqi <= 50:   return "Tốt (Good)"
    if aqi <= 100:  return "Trung bình (Moderate)"
    if aqi <= 150:  return "Không tốt cho nhóm nhạy cảm (USG)"
    if aqi <= 200:  return "Có hại (Unhealthy)"
    if aqi <= 300:  return "Rất có hại (Very Unhealthy)"
    return "Nguy hiểm (Hazardous)"

@udf(StringType())
def dominant_pollutant(aq_pm25, aq_pm10, aq_co, aq_no2, aq_so2, aq_o3):
    """Chất ô nhiễm chính – có AQI cao nhất."""
    c = {"PM2.5":aq_pm25,"PM10":aq_pm10,"CO":aq_co,"NO₂":aq_no2,"SO₂":aq_so2,"O₃":aq_o3}
    v = {k: v for k,v in c.items() if v is not None}
    return max(v, key=v.get) if v else "N/A"

@udf(StringType())
def health_advisory(aqi):
    """Khuyến cáo sức khoẻ theo AQI."""
    if aqi is None: return ""
    if aqi <= 50:  return "Chất lượng tốt – Mọi người có thể hoạt động bình thường ngoài trời."
    if aqi <= 100: return "Chấp nhận được – Người nhạy cảm nên hạn chế hoạt động mạnh ngoài trời."
    if aqi <= 150: return "Nhóm nhạy cảm nên ở trong nhà (người già, trẻ em, bệnh hô hấp)."
    if aqi <= 200: return "Mọi người nên hạn chế ra ngoài, đeo khẩu trang N95 nếu cần thiết."
    if aqi <= 300: return "⚠️ Khẩn cấp: Tránh mọi hoạt động ngoài trời, ở nhà, đóng cửa sổ."
    return "🚨 Nguy hiểm! Toàn bộ dân cư nên ở trong nhà có lọc không khí."


print("⚗️  Tính AQI cho 6 chất ô nhiễm…")
df_enriched = (
    df_clean
    .withColumn("hour",  hour(col("timestamp")))
    .withColumn("day",   dayofmonth(col("timestamp")))
    .withColumn("month", month(col("timestamp")))
    .withColumn("dow",   dayofweek(col("timestamp")))
    .withColumn("AQI_PM25",  aqi_pm25(col("pm25")))
    .withColumn("AQI_PM10",  aqi_pm10(col("pm10")))
    .withColumn("AQI_CO",    aqi_co(col("co")))
    .withColumn("AQI_NO2",   aqi_no2(col("no2")))
    .withColumn("AQI_SO2",   aqi_so2(col("so2")))
    .withColumn("AQI_O3",    aqi_o3(col("o3")))
    .withColumn("AQI_Composite",
        greatest("AQI_PM25","AQI_PM10","AQI_CO","AQI_NO2","AQI_SO2","AQI_O3"))
    .withColumn("AQI_Level", aqi_level(col("AQI_Composite")))
    .withColumn("Dominant_Pollutant",
        dominant_pollutant(col("AQI_PM25"),col("AQI_PM10"),col("AQI_CO"),
                           col("AQI_NO2"), col("AQI_SO2"), col("AQI_O3")))
)

# Z-Score anomaly detection (|z| > 3 → bất thường)
print("   🔍 Phát hiện bất thường (Z-Score Method)…")
stats_z = df_enriched.groupBy("station_id").agg(
    avg("AQI_Composite").alias("mean_aqi"),
    stddev("AQI_Composite").alias("std_aqi"),
)
df_enriched = (
    df_enriched.join(stats_z, on="station_id", how="left")
    .withColumn("Z_Score", F_round((col("AQI_Composite")-col("mean_aqi"))/col("std_aqi"), 3))
    .withColumn("Is_Anomaly", when(F.abs(col("Z_Score")) > 3, "Yes").otherwise("No"))
    .drop("mean_aqi","std_aqi")
    .withColumn("Health_Advisory", health_advisory(col("AQI_Composite")))
)

# Lưu Parquet (enriched)
enriched_path = f"{BASE_DIR}/enriched/air_quality_enriched.parquet"
df_enriched.write.mode("overwrite").parquet(enriched_path)
print(f"   💾 Enriched (Parquet) → {enriched_path}")

# Lưu CSV cho Looker Studio
csv_path = f"{BASE_DIR}/reports/air_quality_report.csv"
df_enriched.write.mode("overwrite").option("header","true").csv(csv_path)
print(f"   💾 Report CSV → {csv_path}")

# ── Auto-test 3 ───────────────────────────────────────────────
for col_name in ["AQI_PM25","AQI_Composite","AQI_Level","Is_Anomaly","Health_Advisory"]:
    assert col_name in df_enriched.columns, f"❌ Thiếu {col_name}!"
assert df_enriched.count() > 0, "❌ DataFrame rỗng!"
print("🎉 PASSED PHẦN 3\n")

⚗️  Tính AQI cho 6 chất ô nhiễm…
   🔍 Phát hiện bất thường (Z-Score Method)…
   💾 Enriched (Parquet) → /content/drive/MyDrive/BigData_AirQuality_VN/enriched/air_quality_enriched.parquet
   💾 Report CSV → /content/drive/MyDrive/BigData_AirQuality_VN/reports/air_quality_report.csv
🎉 PASSED PHẦN 3



In [11]:
# ============================================================
#  PHẦN 4 – ANALYTICS: PHÂN TÍCH THỐNG KÊ ĐA CHIỀU
# ============================================================

print("📊 Phân tích thống kê AQI theo nhiều chiều…")
df_e = df_enriched

# 4.1 Theo trạm
df_by_station = df_e.groupBy("station_id","station_name","city","province","station_type").agg(
    F_round(avg("AQI_Composite"),  2).alias("AQI_Mean"),
    F_round(F_max("AQI_Composite"),2).alias("AQI_Max"),
    F_round(F_min("AQI_Composite"),2).alias("AQI_Min"),
    F_round(stddev("AQI_Composite"),2).alias("AQI_StdDev"),
    F_round(avg("pm25"), 2).alias("PM25_Mean"),
    F_round(avg("pm10"), 2).alias("PM10_Mean"),
    F_round(avg("co"),   3).alias("CO_Mean"),
    F_round(avg("no2"),  2).alias("NO2_Mean"),
    F_round(avg("so2"),  2).alias("SO2_Mean"),
    F_round(avg("o3"),   4).alias("O3_Mean"),
    count("*").alias("Record_Count"),
).orderBy(col("AQI_Mean").desc())

print("\n=== TOP 10 TRẠM Ô NHIỄM NHẤT ===")
df_by_station.show(10, truncate=False)



📊 Phân tích thống kê AQI theo nhiều chiều…

=== TOP 10 TRẠM Ô NHIỄM NHẤT ===
+----------+---------------------+-----------+-----------+------------+--------+-------+-------+----------+---------+---------+-------+--------+--------+-------+------------+
|station_id|station_name         |city       |province   |station_type|AQI_Mean|AQI_Max|AQI_Min|AQI_StdDev|PM25_Mean|PM10_Mean|CO_Mean|NO2_Mean|SO2_Mean|O3_Mean|Record_Count|
+----------+---------------------+-----------+-----------+------------+--------+-------+-------+----------+---------+---------+-------+--------+--------+-------+------------+
|BD01      |Bình Dương – VSIP    |Thủ Dầu Một|Bình Dương |industrial  |157.13  |311    |70     |24.21     |76.84    |126.16   |3.035  |98.9    |65.6    |0.0263 |8784        |
|DNA01     |Đồng Nai – Biên Hòa  |Biên Hòa   |Đồng Nai   |industrial  |157.04  |376    |65     |24.37     |76.76    |126.27   |3.025  |98.74   |65.56   |0.0265 |8784        |
|HCM04     |HCM – Thủ Đức (KCN)  |Hồ Chí Minh|Hồ

In [12]:
# 4.2 Theo tháng (xu hướng thời vụ)
df_by_month = df_e.groupBy("month").agg(
    F_round(avg("AQI_Composite"), 2).alias("AQI_Mean"),
    F_round(avg("pm25"),          2).alias("PM25_Mean"),
    F_round(avg("temperature"),   2).alias("Temp_Mean"),
    F_round(avg("humidity"),      2).alias("Humidity_Mean"),
).orderBy("month")
print("\n=== AQI THEO THÁNG (2024) ===")
df_by_month.show(12)

# 4.3 Theo giờ (diurnal pattern)
df_by_hour = df_e.groupBy("hour").agg(
    F_round(avg("AQI_Composite"), 2).alias("AQI_Mean"),
    F_round(avg("pm25"),          2).alias("PM25_Mean"),
    F_round(avg("o3"),            4).alias("O3_Mean"),
).orderBy("hour")
print("\n=== AQI THEO GIỜ TRONG NGÀY ===")
df_by_hour.show(24)

# 4.4 Phân phối AQI_Level
df_levels = df_e.groupBy("AQI_Level").agg(count("*").alias("Count"))
total_records = df_e.count()
df_levels = df_levels.withColumn(
    "Percentage", F_round(col("Count") / lit(total_records) * 100, 2)
).orderBy(col("Count").desc())
print("\n=== PHÂN PHỐI MỨC CHẤT LƯỢNG KHÔNG KHÍ ===")
df_levels.show(truncate=False)

# 4.5 Tương quan AQI ~ Khí tượng
print("\n=== TƯƠNG QUAN AQI VỚI CÁC YẾU TỐ KHÍ TƯỢNG ===")
for wvar in ["temperature","humidity","wind_speed","pressure","radiation"]:
    c = df_e.select(corr("AQI_Composite", wvar)).first()[0]
    bar = "█" * int(abs(c or 0) * 20)
    sign = "+" if (c or 0) > 0 else "-"
    print(f"   corr(AQI, {wvar:12s}) = {c:+.4f}  {sign}{bar}")

# 4.6 Anomaly summary
df_anomaly = (
    df_e.filter(col("Is_Anomaly") == "Yes")
    .groupBy("station_id","station_name","city","Dominant_Pollutant")
    .agg(count("*").alias("Anomaly_Count"),
         F_round(F_max("AQI_Composite"), 2).alias("Max_AQI"))
    .orderBy(col("Anomaly_Count").desc())
)
print("\n=== TOP 10 TRẠM BẤT THƯỜNG NHIỀU NHẤT ===")
df_anomaly.show(10, truncate=False)



=== AQI THEO THÁNG (2024) ===
+-----+--------+---------+---------+-------------+
|month|AQI_Mean|PM25_Mean|Temp_Mean|Humidity_Mean|
+-----+--------+---------+---------+-------------+
|    1|  138.04|    55.56|    22.96|        69.96|
|    2|  138.18|    55.66|    24.13|        62.47|
|    3|   137.9|    55.58|    25.75|        56.99|
|    4|  120.11|     45.2|    29.43|        55.06|
|    5|  106.05|    37.81|    28.73|        57.07|
|    6|  106.15|     37.8|    28.76|        62.48|
|    7|  105.98|     37.9|    27.93|        70.02|
|    8|  106.01|    37.87|    28.29|         77.5|
|    9|   106.1|    37.88|    27.17|        83.03|
|   10|  106.17|    37.87|    26.15|        84.99|
|   11|  124.02|    48.18|    25.74|        82.99|
|   12|  137.96|    55.61|    22.47|        77.43|
+-----+--------+---------+---------+-------------+


=== AQI THEO GIỜ TRONG NGÀY ===
+----+--------+---------+-------+
|hour|AQI_Mean|PM25_Mean|O3_Mean|
+----+--------+---------+-------+
|   0|  111.74|  

In [13]:
# 4.7 Top ngày ô nhiễm
df_by_day = (
    df_e.withColumn("date", date_format("timestamp","yyyy-MM-dd"))
    .groupBy("date","province")
    .agg(F_round(avg("AQI_Composite"),2).alias("AQI_Daily"),
         F_round(F_max("AQI_Composite"),2).alias("AQI_Max"))
    .orderBy(col("AQI_Max").desc())
)
print("\n=== TOP 15 NGÀY Ô NHIỄM NHẤT ===")
df_by_day.show(15, truncate=False)

# Lưu analytics
for df_out, name in [
    (df_by_station, "by_station"),
    (df_by_month,   "by_month"),
    (df_by_hour,    "by_hour"),
    (df_levels,     "aqi_levels"),
    (df_anomaly,    "anomalies"),
]:
    df_out.write.mode("overwrite").option("header","true").csv(f"{BASE_DIR}/analytics/{name}.csv")
print("   💾 Tất cả analytics đã lưu vào thư mục analytics/")

# ── Auto-test 4 ───────────────────────────────────────────────
assert df_by_station.count() == len(STATIONS), "❌ Số trạm không khớp!"
assert df_by_month.count() == 12,              "❌ Thiếu tháng!"
assert df_by_hour.count()  == 24,              "❌ Thiếu giờ!"
print("🎉 PASSED PHẦN 4\n")


=== TOP 15 NGÀY Ô NHIỄM NHẤT ===
+----------+----------+---------+-------+
|date      |province  |AQI_Daily|AQI_Max|
+----------+----------+---------+-------+
|2024-03-25|Đồng Nai  |177.33   |376    |
|2024-01-08|Hà Nội    |137.06   |361    |
|2024-02-11|Quảng Ninh|176.88   |349    |
|2024-01-23|Hà Nội    |134.32   |345    |
|2024-02-07|Quảng Ninh|180.31   |336    |
|2024-03-25|Hải Phòng |156.94   |332    |
|2024-02-04|Hà Nội    |136.58   |324    |
|2024-03-28|Quảng Ninh|175.1    |321    |
|2024-12-24|Hà Nội    |136.08   |319    |
|2024-01-03|Bình Dương|171.21   |319    |
|2024-02-23|Hà Nội    |134.8    |318    |
|2024-12-11|Hà Nội    |135.24   |317    |
|2024-02-24|Hà Nội    |135.92   |315    |
|2024-03-03|Hà Nội    |136.65   |315    |
|2024-11-02|Quảng Ninh|175.52   |312    |
+----------+----------+---------+-------+
only showing top 15 rows
   💾 Tất cả analytics đã lưu vào thư mục analytics/
🎉 PASSED PHẦN 4



In [14]:
# ============================================================
#  PHẦN 4 – EXPORT (đọc lại từ CSV đã lưu ở Phần 4)
# ============================================================

BASE_DIR = "/content/drive/MyDrive/BigData_AirQuality_VN"
EXPORT_PATH = f"{BASE_DIR}/analytics/enriched_export"
LONG_PATH   = f"{BASE_DIR}/analytics/enriched_long"

# Đọc lại từ CSV by_station đã lưu để rebuild df_e tạm
print("🔄 Đọc lại dữ liệu từ các file CSV Phần 4 đã lưu...")

df_by_station = spark.read.option("header","true").csv(f"{BASE_DIR}/analytics/by_station.csv")
df_by_month   = spark.read.option("header","true").csv(f"{BASE_DIR}/analytics/by_month.csv")
df_by_hour    = spark.read.option("header","true").csv(f"{BASE_DIR}/analytics/by_hour.csv")
df_levels     = spark.read.option("header","true").csv(f"{BASE_DIR}/analytics/aqi_levels.csv")
df_anomaly    = spark.read.option("header","true").csv(f"{BASE_DIR}/analytics/anomalies.csv")

print(f"   ✅ by_station: {df_by_station.count()} trạm")
print(f"   ✅ by_month:   {df_by_month.count()} tháng")
print(f"   ✅ by_hour:    {df_by_hour.count()} giờ")
print(f"   ✅ aqi_levels: {df_levels.count()} mức")
print(f"   ✅ anomalies:  {df_anomaly.count()} trạm bất thường")
print("🎉 EXPORT XONG\n")

🔄 Đọc lại dữ liệu từ các file CSV Phần 4 đã lưu...
   ✅ by_station: 49 trạm
   ✅ by_month:   12 tháng
   ✅ by_hour:    24 giờ
   ✅ aqi_levels: 6 mức
   ✅ anomalies:  64 trạm bất thường
🎉 EXPORT XONG



In [15]:
# ============================================================
#  BỔ SUNG: CHUẨN BỊ DỮ LIỆU ĐỂ CHẠY PHẦN 5
# ============================================================
from pyspark.sql.functions import col, avg, round as F_round

print("🧪 Đang trích xuất thông số trung bình từ dữ liệu mô phỏng...")

# 1. Tính toán PM2.5 trung bình từ kết quả by_station (Phần 4)
# Vì các cột từ CSV được đọc lên ở dạng String, chúng ta cần cast về Double
sim_summary = df_by_station.select(
    F_round(avg(col("PM25_Mean").cast("double")), 2).alias("sim_pm25_avg"),
    F_round(avg(col("AQI_Mean").cast("double")), 2).alias("sim_aqi_avg")
).collect()

# Gán giá trị vào biến để Phần 5 sử dụng
sim_pm25_mean = sim_summary[0]["sim_pm25_avg"]
sim_aqi_mean = sim_summary[0]["sim_aqi_avg"]

print(f"   📊 Giá trị mô phỏng chuẩn bị: PM2.5 = {sim_pm25_mean}, AQI = {sim_aqi_mean}")

# 2. Tạo file JSON "giả lập thực tế" nếu chưa có dữ liệu từ API
# Mục đích: Đảm bảo Phần 5 không bị lỗi 'File Not Found'
import os
import json

openaq_path = f"{BASE_DIR}/raw/openaq_realdata.json"

if not os.path.exists(openaq_path):
    print("   ⚠️ Không tìm thấy file openaq_realdata.json, đang tạo dữ liệu mẫu...")
    # Tạo 1 vài bản ghi mẫu có cấu trúc giống OpenWeather API để test logic Phần 5
    sample_real_data = [
        {
            "station_id": "REAL_01",
            "pm2_5": 35.5,
            "pm10": 42.1,
            "co": 0.8,
            "no2": 20.2,
            "aqi": 3,
            "timestamp": 1711872000
        },
        {
            "station_id": "REAL_02",
            "pm2_5": 45.2,
            "pm10": 55.0,
            "co": 1.1,
            "no2": 35.5,
            "aqi": 4,
            "timestamp": 1711875600
        }
    ]
    with open(openaq_path, "w", encoding="utf-8") as f:
        json.dump(sample_real_data, f, indent=4)
    print(f"   ✅ Đã lưu file mẫu tại: {openaq_path}")
else:
    print(f"   ✅ Đã xác nhận file dữ liệu thực tế tại: {openaq_path}")

print("\n🚀 Đã sẵn sàng để chạy PHẦN 5: SO SÁNH THỰC TẾ")

🧪 Đang trích xuất thông số trung bình từ dữ liệu mô phỏng...
   📊 Giá trị mô phỏng chuẩn bị: PM2.5 = 45.22, AQI = 119.34
   ✅ Đã xác nhận file dữ liệu thực tế tại: /content/drive/MyDrive/BigData_AirQuality_VN/raw/openaq_realdata.json

🚀 Đã sẵn sàng để chạy PHẦN 5: SO SÁNH THỰC TẾ


In [16]:
# ============================================================
#  PHẦN 5 – TÍCH HỢP OPENAQ THỰC TẾ & SO SÁNH ĐỐI CHIẾU
# ============================================================
from pyspark.sql.functions import expr, col, avg, lit, round as F_round
import os

print("🔗 Đang chuẩn bị dữ liệu đối chiếu cho Phần 5...")

# --- 1. LẤY SỐ LIỆU MÔ PHỎNG (Simulated Data) ---
# Ưu tiên đọc từ Parquet của Phần 3 để giữ độ chính xác cao nhất
ENRICHED_PARQUET = f"{BASE_DIR}/enriched/air_quality_enriched.parquet"

try:
    if os.path.exists(ENRICHED_PARQUET):
        df_sim = spark.read.parquet(ENRICHED_PARQUET)
        sim_pm25_mean = df_sim.agg(F_round(avg("pm25"), 2)).first()[0]
        print(f"   ✅ Lấy số liệu từ Parquet: PM2.5 Mean = {sim_pm25_mean}")
    else:
        # Nếu không có Parquet, sử dụng df_by_station (CSV) vừa nạp ở bước trước
        sim_pm25_mean = df_by_station.select(avg(col("PM25_Mean").cast("double"))).first()[0]
        sim_pm25_mean = round(float(sim_pm25_mean), 2)
        print(f"   ✅ Lấy số liệu từ CSV Export: PM2.5 Mean = {sim_pm25_mean}")
except Exception as e:
    sim_pm25_mean = 0
    print(f"   ⚠️ Cảnh báo: Không thể tính số liệu mô phỏng ({e})")


# --- 2. XỬ LÝ DỮ LIỆU THỰC TẾ (Real-world Data từ API) ---
openaq_path = f"{BASE_DIR}/raw/openaq_realdata.json"

if os.path.exists(openaq_path):
    print(f"📂 Đang đọc dữ liệu thực tế từ: {openaq_path}")
    df_oaq_raw = spark.read.json(openaq_path)

    if df_oaq_raw.count() > 0:
        # Chuẩn hóa cột để tránh lỗi Schema
        target_cols = {"pm2_5": "pm25", "pm10": "pm10", "co": "co", "no2": "no2", "aqi": "aqi"}
        for c_raw in target_cols.keys():
            if c_raw not in df_oaq_raw.columns:
                df_oaq_raw = df_oaq_raw.withColumn(c_raw, lit(None).cast("double"))

        # Chuyển đổi sang dạng Long Format để phân tích đa chiều
        df_oaq_long = df_oaq_raw.select(
            col("station_id") if "station_id" in df_oaq_raw.columns else lit("VN_Station").alias("station_id"),
            expr("""stack(5,
                'pm25', cast(pm2_5 as double),
                'pm10', cast(pm10 as double),
                'co',   cast(co as double),
                'no2',  cast(no2 as double),
                'aqi',  cast(aqi as double)) as (parameter, value)""")
        ).filter("value IS NOT NULL")

        # Tính trung bình thực tế
        oaq_stats = df_oaq_long.filter(col("parameter") == "pm25").agg(avg("value")).first()
        oaq_pm25_mean = round(oaq_stats[0], 2) if (oaq_stats and oaq_stats[0]) else 0

        # --- 3. HIỂN THỊ KẾT QUẢ SO SÁNH ---
        print("\n" + "="*50)
        print("📊 TỔNG KẾT SO SÁNH DỮ LIỆU (MÔ PHỎNG VS THỰC TẾ)")
        print("="*50)
        print(f"📍 Chỉ số PM2.5 trung bình (Mô phỏng): {sim_pm25_mean:>10} µg/m³")
        print(f"📍 Chỉ số PM2.5 trung bình (Thực tế):  {oaq_pm25_mean:>10} µg/m³")

        diff = round(abs(sim_pm25_mean - oaq_pm25_mean), 2)
        accuracy = round(100 - (diff/max(sim_pm25_mean, 1) * 100), 2)

        print(f"📉 Chênh lệch: {diff} µg/m³")
        print(f"🎯 Độ tương đồng mô hình: {max(0, accuracy)}%")
        print("="*50)

        # Lưu kết quả cuối cùng để vẽ biểu đồ
        output_final = f"{BASE_DIR}/analytics/final_comparison.csv"
        df_oaq_long.write.mode("overwrite").option("header","true").csv(output_final)
        print(f"✅ Đã lưu kết quả so sánh tại: {output_final}")

    else:
        print("⚠️ File JSON thực tế không có dữ liệu bản ghi.")
else:
    print(f"❌ Không tìm thấy file API thực tế tại: {openaq_path}")
    print("💡 Gợi ý: Hãy kiểm tra lại Bước 1.5 để đảm bảo API đã tải dữ liệu về.")

print("\n🎉 HOÀN THÀNH TOÀN BỘ PIPELINE!")

🔗 Đang chuẩn bị dữ liệu đối chiếu cho Phần 5...
   ✅ Lấy số liệu từ Parquet: PM2.5 Mean = 45.22
📂 Đang đọc dữ liệu thực tế từ: /content/drive/MyDrive/BigData_AirQuality_VN/raw/openaq_realdata.json

📊 TỔNG KẾT SO SÁNH DỮ LIỆU (MÔ PHỎNG VS THỰC TẾ)
📍 Chỉ số PM2.5 trung bình (Mô phỏng):      45.22 µg/m³
📍 Chỉ số PM2.5 trung bình (Thực tế):           0 µg/m³
📉 Chênh lệch: 45.22 µg/m³
🎯 Độ tương đồng mô hình: 0%
✅ Đã lưu kết quả so sánh tại: /content/drive/MyDrive/BigData_AirQuality_VN/analytics/final_comparison.csv

🎉 HOÀN THÀNH TOÀN BỘ PIPELINE!


In [17]:
# ============================================================
#  PHẦN 6 – VISUALIZATION: LOOKER STUDIO
# ============================================================
"""
HƯỚNG DẪN VẼ DASHBOARD LOOKER STUDIO:

1. Mở Google Drive → BigData_AirQuality_VN/reports/
   Tải file .csv về máy (chọn file part-*.csv không có tên SUCCESS)

2. Vào https://lookerstudio.google.com/ → Tạo Báo cáo trống

3. Thêm nguồn dữ liệu: "Tải tệp lên" → chọn file CSV vừa tải

4. Vẽ tối thiểu 5 biểu đồ:
   a) Thẻ điểm (Scorecard)   : AQI_Composite trung bình
   b) Biểu đồ cột (Bar)      : AQI_Mean theo province (màu theo station_type)
   c) Biểu đồ tròn (Pie)     : Phân phối AQI_Level
   d) Chuỗi thời gian (Time) : AQI_Composite theo timestamp
      + đường tham chiếu AQI = 100
   e) Bảng (Table)           : Top 10 trạm AQI_Max cao nhất

5. Chia sẻ → "Bất kỳ ai có liên kết đều có thể xem"
6. Dán link vào biến bên dưới
"""

LOOKER_STUDIO_LINK = "https://lookerstudio.google.com/reporting/YOUR_REPORT_ID"

assert LOOKER_STUDIO_LINK.startswith("https://lookerstudio.google.com/reporting/"), \
    "❌ Link Looker Studio không hợp lệ!"
print(f"\n📊 Looker Studio: {LOOKER_STUDIO_LINK}")


📊 Looker Studio: https://lookerstudio.google.com/reporting/YOUR_REPORT_ID


In [18]:
# ============================================================
#  TỔNG KẾT
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════╗
║    PIPELINE HOÀN THÀNH – BIG DATA CHẤT LƯỢNG KHÔNG KHÍ VN    ║
╠══════════════════════════════════════════════════════════════╣
║  Nguồn dữ liệu (100% tự động qua API):                       ║
║    • Open-Meteo Archive API – 5 thành phố × 1 năm            ║
║    • OpenAQ API v2          – 4 chất ô nhiễm thực tế tại VN  ║
║    • Dataset tổng hợp       – 50 trạm × 365 ngày × 24h       ║
║                                                              ║
║  Quy mô: 438,000+ bản ghi xử lý bằng PySpark                 ║
║  Lưu trữ: Parquet (clean) + CSV (Looker Studio)              ║
║  Chất ô nhiễm: PM2.5, PM10, CO, NO₂, SO₂, O₃                 ║
║  Tiêu chuẩn: US EPA AQI – Piecewise Linear Interpolation     ║
║                                                              ║
║  Pipeline:                                                   ║
║  [API] → [Raw JSON] → [PySpark Clean] → [AQI Enriched]       ║
║        → [Parquet] → [Analytics CSV] → [Looker Studio]       ║
╚══════════════════════════════════════════════════════════════╝
""")

spark.stop()
print("✅ SparkSession đã dừng. Đồ án hoàn thành!")


╔══════════════════════════════════════════════════════════════╗
║    PIPELINE HOÀN THÀNH – BIG DATA CHẤT LƯỢNG KHÔNG KHÍ VN    ║
╠══════════════════════════════════════════════════════════════╣
║  Nguồn dữ liệu (100% tự động qua API):                       ║
║    • Open-Meteo Archive API – 5 thành phố × 1 năm            ║
║    • OpenAQ API v2          – 4 chất ô nhiễm thực tế tại VN  ║
║    • Dataset tổng hợp       – 50 trạm × 365 ngày × 24h       ║
║                                                              ║
║  Quy mô: 438,000+ bản ghi xử lý bằng PySpark                 ║
║  Lưu trữ: Parquet (clean) + CSV (Looker Studio)              ║
║  Chất ô nhiễm: PM2.5, PM10, CO, NO₂, SO₂, O₃                 ║
║  Tiêu chuẩn: US EPA AQI – Piecewise Linear Interpolation     ║
║                                                              ║
║  Pipeline:                                                   ║
║  [API] → [Raw JSON] → [PySpark Clean] → [AQI Enriched]       ║
║        → [Parquet] → [